In [1]:
# Cài đặt các thư viện bắt buộc cho Pipeline và ép cập nhật torchao
!pip install -q -U "torchao>0.16.0"
!pip install -q open_clip_torch
!pip install -q evaluate
!pip install -q rouge_score
!pip install -q bert_score
!pip install -q datasets
!pip install -q accelerate
!pip install -q scikit-learn
!pip install -q transformers qwen-vl-utils
!pip install evaluate rouge_score bert_score scikit-learn
!pip install -q bitsandbytes accelerate
print("✅ Đã cài đặt xong tất cả các thư viện cần thiết và cập nhật torchao!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 50.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 28.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 974.8 kB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 63.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.1 MB/s eta 0:00:00:00:0100:01
✅ Đã cài đặt xong tất cả các thư viện cần thiết và cập nhật torchao!


In [2]:
import os
import gc
import time
import torch
import re
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from datasets import load_dataset
import transformers
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
import bitsandbytes as bnb
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score
import evaluate
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm
import warnings
import ctypes
import urllib.request
import zipfile

warnings.filterwarnings("ignore", category=UserWarning)
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ==========================================
# 0. THIẾT LẬP MÔI TRƯỜNG KAGGLE & TỐI ƯU CUDNN
# ==========================================
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.8"
torch.backends.cudnn.benchmark = True 

runName = "FedMVQA_SLAKE_Qwen2VL2B_FFT_448_Alpha1_R1_E2_3Clients"

kaggleBaseDir = "/kaggle/working/Fed-MVQA-Project"
dataDir = os.path.join(kaggleBaseDir, "data", "slake")
logDir = os.path.join(kaggleBaseDir, "logs")
checkpointDir = os.path.join(kaggleBaseDir, "checkpoints", runName)

os.makedirs(dataDir, exist_ok=True)
os.makedirs(logDir, exist_ok=True)
os.makedirs(checkpointDir, exist_ok=True)

logPath = os.path.join(logDir, f"{runName}_log.csv")
checkpointPath = os.path.join(checkpointDir, "best_fed_mvqa_model.pt")

# ==========================================
# 1. CÔNG CỤ QUẢN LÝ BỘ NHỚ CẤP THẤP
# ==========================================
def force_garbage_collect():
    """Hút cạn sạch bộ nhớ rác của Python và C++"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    try:
        libc = ctypes.CDLL("libc.so.6")
        libc.malloc_trim(0)
    except Exception:
        pass

def getPeakReservedMemory():
    if torch.cuda.is_available():
        reservedGb = torch.cuda.memory_reserved() / (1024 ** 3)
        return f"{reservedGb:.2f} GB"
    return "0.00 GB"

def calculateTrainableParameters(model):
    trainableParams = sum(p.numel() for p in model.parameters() if p.requires_grad)
    allParams = sum(p.numel() for p in model.parameters())
    percentage = 100 * trainableParams / allParams
    return trainableParams, allParams, percentage

def calculateCommunicationMetrics(totalParams, numClients):
    fullModelBytes = totalParams * 2 
    fullRoundCostMb = (fullModelBytes * 2 * numClients) / (1024 * 1024)
    return fullRoundCostMb

# ==========================================
# 2. KHỞI TẠO MÔ HÌNH TOÀN CỤC (FFT 448x448, FLOAT16)
# ==========================================
print("--- KHỞI TẠO HẠ TẦNG FED-MVQA (FFT 448x448, FLOAT16 - 3 CLIENTS) ---")
force_garbage_collect()

processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    min_pixels=256 * 28 * 28,
    max_pixels=256 * 28 * 28 
)

baseModel = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.float16, 
    device_map={"": 0}, 
    attn_implementation="sdpa",
    low_cpu_mem_usage=True
).cuda()

baseModel.config.use_cache = False
baseModel.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

for param in baseModel.parameters():
    param.requires_grad = True

trainableParams, totalModelParams, trainPct = calculateTrainableParameters(baseModel)
print(f"[Metrics] Trainable Params (FFT): {trainableParams:,} (~{trainPct:.3f}%)")

globalWeights = {k: v.detach().cpu().clone() for k, v in baseModel.state_dict().items()}
totalParamsCount = totalModelParams
force_garbage_collect()

# ==========================================
# 3. CHUẨN BỊ VÀ PHÂN MẢNH DỮ LIỆU SLAKE (ALPHA = 1.0)
# ==========================================
print("\n--- CHUẨN BỊ DỮ LIỆU SLAKE ENGLISH (NEAR-IID α=1.0) ---")

if not os.path.exists("./imgs"):
    print(" [Hệ thống] Đang tải file ảnh imgs.zip từ Hugging Face...")
    urllib.request.urlretrieve("https://huggingface.co/datasets/BoKelvin/SLAKE/resolve/main/imgs.zip", "imgs.zip")
    with zipfile.ZipFile("imgs.zip", 'r') as zipRef:
        zipRef.extractall("./")
    os.remove("imgs.zip")
    print(" [Hệ thống] Đã chuẩn bị xong thư mục ảnh cục bộ!")

def loadSlakeImage(imgName):
    imgPath = os.path.join(".", "imgs", imgName.replace("imgs/", ""))
    return Image.open(imgPath).convert("RGB")

dataset = load_dataset("BoKelvin/SLAKE", cache_dir=dataDir)

englishTrain = dataset["train"].filter(lambda x: x["q_lang"] == "en")
englishTest = dataset["test"].filter(lambda x: x["q_lang"] == "en")

shuffledTrain = englishTrain.shuffle(seed=42)
testDataset = englishTest.shuffle(seed=42).select(range(min(500, len(englishTest))))

def partitionDataNonIidDirichlet(datasetInput, numClients=3, alpha=1.0):
    labels = [str(item["answer"]).strip().lower() for item in datasetInput]
    uniqueLabels = list(set(labels))
    clientIndices = {i: [] for i in range(numClients)}
    for label in uniqueLabels:
        labelIdxs = [i for i, l in enumerate(labels) if l == label]
        np.random.shuffle(labelIdxs)
        proportions = np.random.dirichlet([alpha] * numClients)
        splits = np.split(labelIdxs, (np.cumsum(proportions)[:-1] * len(labelIdxs)).astype(int))
        for i in range(numClients):
            clientIndices[i].extend(splits[i].tolist())
            
    clientDatasets = []
    for i in range(numClients):
        np.random.shuffle(clientIndices[i])
        clientDatasets.append(datasetInput.select(clientIndices[i]))
    return clientDatasets

clientDatasets = partitionDataNonIidDirichlet(shuffledTrain, numClients=3, alpha=1.0)
datasetClientA = clientDatasets[0]
datasetClientB = clientDatasets[1]
datasetClientC = clientDatasets[2]
dataSizes = [len(datasetClientA), len(datasetClientB), len(datasetClientC)]

print(f"Dữ liệu Client A: {dataSizes[0]} mẫu")
print(f"Dữ liệu Client B: {dataSizes[1]} mẫu")
print(f"Dữ liệu Client C: {dataSizes[2]} mẫu")

headerTokens = processor.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)
headerLen = len(headerTokens)

systemInstruction = (
    "You are an expert medical diagnostic assistant. Your task is to answer clinical questions "
    "based EXCLUSIVELY on the provided medical image.\n"
    "Strictly follow these rules:\n"
    "1. Analyze the provided [Target Case Image] and the [Question].\n"
    "2. Provide a concise and definitive answer (1-3 words if possible)."
)

def customCollateFn(batch):
    texts, images = [], []
    for item in batch:
        img = loadSlakeImage(item["img_name"])
        q = item["question"]
        a = str(item["answer"]).strip()
        
        messages = [
            {"role": "system", "content": [{"type": "text", "text": systemInstruction}]},
            {"role": "user", "content": [
                {"type": "text", "text": "[Target Case Image]\n"},
                {"type": "image", "image": img}, 
                {"type": "text", "text": f"\nCurrent Question: {q}\n\nBased on the visual evidence, what is the correct answer for the current case?\nFinal Answer:"}
            ]},
            {"role": "assistant", "content": [{"type": "text", "text": a}]}
        ]
        textPrompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(textPrompt)
        images.append(img)

    inputs = processor(text=texts, images=images, return_tensors="pt", padding=True)
    labels = inputs["input_ids"].clone()
    for i in range(labels.shape[0]):
        tokenIds = labels[i].tolist()
        if processor.tokenizer.pad_token_id is not None:
            labels[i][labels[i] == processor.tokenizer.pad_token_id] = -100
        matchIdx = -1
        for j in range(len(tokenIds) - headerLen + 1):
            if tokenIds[j:j+headerLen] == headerTokens:
                matchIdx = j + headerLen 
                break
        if matchIdx != -1:
            labels[i, :matchIdx] = -100 
    inputs["labels"] = labels
    return inputs

batchSize = 1
trainLoaderA = DataLoader(datasetClientA, batch_size=batchSize, shuffle=True, collate_fn=customCollateFn, num_workers=0, pin_memory=False)
trainLoaderB = DataLoader(datasetClientB, batch_size=batchSize, shuffle=True, collate_fn=customCollateFn, num_workers=0, pin_memory=False)
trainLoaderC = DataLoader(datasetClientC, batch_size=batchSize, shuffle=True, collate_fn=customCollateFn, num_workers=0, pin_memory=False)

# ==========================================
# 4. HÀM HUẤN LUYỆN CỤC BỘ (PAGED OPTIMIZER)
# ==========================================
def trainLocalClient(model, dataLoader, clientId, roundIdx, epochs=2, learningRate=1e-5): 
    model.train()
    
    decayFactor = 0.9 ** (roundIdx - 1)
    activeLr = learningRate * decayFactor
    
    optimizer = bnb.optim.PagedAdamW8bit(model.parameters(), lr=activeLr, weight_decay=0.01)
    accumulationSteps = 4  
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs * len(dataLoader) // accumulationSteps)
    
    print(f" [Client {clientId}] Bắt đầu FFT 448x448 (PagedAdamW8bit, LR={activeLr:.2e})...")
    startTime = time.time()
    totalLoss = 0
    
    for epoch in range(epochs):
        optimizer.zero_grad(set_to_none=True)
        progressLoop = tqdm(dataLoader, desc=f"Client {clientId} | Epoch {epoch + 1}/{epochs}", leave=True, dynamic_ncols=True)
        
        for step, batch in enumerate(progressLoop):
            batch = {k: v.cuda(non_blocking=True) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            
            with torch.amp.autocast("cuda", dtype=torch.float16):
                outputs = model(**batch)
                loss = outputs.loss / accumulationSteps
            
            loss.backward()
            
            if (step + 1) % accumulationSteps == 0 or (step + 1) == len(dataLoader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                
            stepLoss = loss.item() * accumulationSteps
            totalLoss += stepLoss
            
            currentLr = scheduler.get_last_lr()[0]
            progressLoop.set_postfix(Loss=f"{stepLoss:.4f}", LR=f"{currentLr:.2e}")
            
            del outputs, loss, batch
            if step % 50 == 0:  
                force_garbage_collect()
            
        avgLoss = totalLoss / len(dataLoader)
        finalLoss = avgLoss
        print(f"      Epoch {epoch + 1} hoàn tất | Avg Loss: {avgLoss:.4f}")
    
    trainDuration = time.time() - startTime
    peakVram = getPeakReservedMemory()
    
    del optimizer, scheduler
    force_garbage_collect()
    return trainDuration, finalLoss, peakVram

# ==========================================
# 5. HÀM ĐÁNH GIÁ (ĐÃ TỐI ƯU CLOSED QUESTIONS)
# ==========================================
def cleanOpenAnswer(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    stopwords = {'a', 'an', 'the', 'is', 'are', 'was', 'were', 'it', 'there', 'shows', 'showing', 'of', 'in', 'on', 'to'}
    words = text.split()
    words = [w for w in words if w not in stopwords]
    return " ".join(words)

def evaluateGlobalModel(model, datasetInput, processorInput):
    model.eval()
    print(f"\n [Server] Đang đánh giá trên tập Test ({len(datasetInput)} mẫu)...")
    
    openPreds, openRefs, closePreds, closeRefs = [], [], [], []
    overallCorrect, inferenceTimes = 0, []
    
    yesKeywords = ['yes', 'correct', 'true', 'positive', 'present', 'visible']
    noKeywords = ['no', 'incorrect', 'false', 'negative', 'absent', 'normal', 'none']
    
    evalLoop = tqdm(datasetInput, desc="Evaluating", leave=False, dynamic_ncols=True)
    
    for item in evalLoop:
        img = loadSlakeImage(item["img_name"])
        q = item["question"]
        gtAnswer = str(item["answer"]).strip().lower()
        ansType = str(item.get("answer_type", "")).strip().upper()
        if ansType == "": ansType = "CLOSED" if gtAnswer in ["yes", "no"] else "OPEN"
        
        messages = [
            {"role": "system", "content": [{"type": "text", "text": systemInstruction}]},
            {"role": "user", "content": [
                {"type": "text", "text": "[Target Case Image]\n"},
                {"type": "image", "image": img}, 
                {"type": "text", "text": f"\nCurrent Question: {q}\n\nBased on the visual evidence, what is the correct answer for the current case?\nFinal Answer:"}
            ]}
        ]
        
        textPrompt = processorInput.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processorInput(text=[textPrompt], images=[img], return_tensors="pt", padding=True)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        startTime = time.time()
        with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
            generatedIds = model.generate(**inputs, max_new_tokens=15, do_sample=False)
        inferenceTimes.append(time.time() - startTime)
        
        generatedIdsTrimmed = [outIds[len(inIds):] for inIds, outIds in zip(inputs["input_ids"], generatedIds)]
        aiAnswerRaw = processorInput.batch_decode(generatedIdsTrimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip().lower()
        aiAnswer = aiAnswerRaw.replace("final answer:", "").strip()
        
        aiAnswerClean = cleanOpenAnswer(aiAnswer)
        gtAnswerClean = cleanOpenAnswer(gtAnswer)
        
        if ansType == "CLOSED":
            if gtAnswer in ['yes', 'no']:
                hasYes = any(word in aiAnswer for word in yesKeywords)
                hasNo = any(word in aiAnswer for word in noKeywords)
                
                if hasYes and not hasNo: predClean = 'yes'
                elif hasNo and not hasYes: predClean = 'no'
                else:
                    firstYesPos = aiAnswer.find('yes') if 'yes' in aiAnswer else 9999
                    firstNoPos = aiAnswer.find('no') if 'no' in aiAnswer else 9999
                    predClean = 'yes' if firstYesPos < firstNoPos else ('no' if firstNoPos < firstYesPos else 'unknown')
            else:
                predClean = aiAnswerClean
                
            closePreds.append(predClean)
            closeRefs.append(gtAnswerClean if gtAnswer not in ['yes', 'no'] else gtAnswer)
            
            if predClean == (gtAnswerClean if gtAnswer not in ['yes', 'no'] else gtAnswer): 
                overallCorrect += 1
        else:
            openPreds.append(aiAnswerClean); openRefs.append(gtAnswerClean)
            if aiAnswerClean == gtAnswerClean: 
                overallCorrect += 1
                
        del inputs, generatedIds, generatedIdsTrimmed
        torch.cuda.empty_cache()

    res = {"overallAccuracy": overallCorrect / len(datasetInput) if len(datasetInput) > 0 else 0, "avgInferenceTime": sum(inferenceTimes) / len(inferenceTimes)}
    if closeRefs: res["Closed"] = {"ACC": accuracy_score(closeRefs, closePreds), "F1": f1_score(closeRefs, closePreds, average='weighted', zero_division=0), "count": len(closeRefs)}
    if openRefs:
        bertRes = evaluate.load("bertscore").compute(predictions=openPreds, references=openRefs, lang="en")
        res["Open"] = {"BLEU-1": evaluate.load("bleu").compute(predictions=openPreds, references=[[r] for r in openRefs], max_order=1)["bleu"], "ROUGE-L": evaluate.load("rouge").compute(predictions=openPreds, references=[[r] for r in openRefs])["rougeL"], "BERTScore": sum(bertRes["f1"]) / len(bertRes["f1"]), "count": len(openRefs)}
    
    return res

# ==========================================
# 6. VÒNG LẶP HỌC LIÊN KẾT CHÍNH (IN-PLACE MUTATION TỐI ĐA)
# ==========================================
maxRounds = 1  
localEpochs = 2
cumulativeFullGb = 0.0
performanceLog = []
peakVramRecorded = "0.00 GB"
totalSamples = sum(dataSizes)

print(f"\n=== BẮT ĐẦU HUẤN LUYỆN FED-MVQA ({maxRounds} ROUNDS, {localEpochs} EPOCH/CLIENT, ALPHA=1.0, 448x448, 3 CLIENTS) ===")

for roundIdx in range(1, maxRounds + 1):
    print(f"\n" + "="*50)
    print(f"--- COMMUNICATION ROUND {roundIdx} ---")
    print("="*50)
    latestTrainTimes = []
    
    print("\n [Server] Đang thiết lập cấu trúc In-Place Accumulation...")
    backupWeights = {k: v.clone() for k, v in globalWeights.items()}
    
    for k in globalWeights.keys():
        if globalWeights[k].dtype.is_floating_point:
            globalWeights[k].zero_()
            
    # --------- Client A ---------
    baseModel.load_state_dict(backupWeights, strict=False)
    timeA, lossA, vramA = trainLocalClient(baseModel, trainLoaderA, clientId="A", roundIdx=roundIdx, epochs=localEpochs, learningRate=1e-5)
    latestTrainTimes.append(timeA)
    
    weightRatioA = dataSizes[0] / totalSamples
    for k, v in baseModel.state_dict().items():
        if v.dtype.is_floating_point:
            globalWeights[k].add_(v.detach().cpu().float() * weightRatioA)
    force_garbage_collect()
    
    # --------- Client B ---------
    baseModel.load_state_dict(backupWeights, strict=False)
    timeB, lossB, vramB = trainLocalClient(baseModel, trainLoaderB, clientId="B", roundIdx=roundIdx, epochs=localEpochs, learningRate=1e-5)
    latestTrainTimes.append(timeB)
    
    weightRatioB = dataSizes[1] / totalSamples
    for k, v in baseModel.state_dict().items():
        if v.dtype.is_floating_point:
            globalWeights[k].add_(v.detach().cpu().float() * weightRatioB)
    force_garbage_collect()
    
    # --------- Client C ---------
    baseModel.load_state_dict(backupWeights, strict=False)
    timeC, lossC, vramC = trainLocalClient(baseModel, trainLoaderC, clientId="C", roundIdx=roundIdx, epochs=localEpochs, learningRate=1e-5)
    latestTrainTimes.append(timeC)
    peakVramRecorded = vramC
    
    weightRatioC = dataSizes[2] / totalSamples
    for k, v in baseModel.state_dict().items():
        if v.dtype.is_floating_point:
            globalWeights[k].add_(v.detach().cpu().float() * weightRatioC)
    force_garbage_collect()
    
    # --------- Hoàn tất Aggregation ---------
    print("\n [Server] Hoàn tất In-Place Aggregation. Ép kiểu về Float16...")
    for k, v in globalWeights.items():
        if v.dtype.is_floating_point:
            globalWeights[k] = v.to(torch.float16)
            
    del backupWeights
    force_garbage_collect()
    
    baseModel.load_state_dict(globalWeights, strict=False)
    force_garbage_collect()
    
    evalResults = evaluateGlobalModel(baseModel, testDataset, processor)
    
    roundAccuracy = evalResults["overallAccuracy"] * 100
    roundAccClosed = evalResults.get("Closed", {}).get("ACC", 0) * 100
    roundF1Closed = evalResults.get("Closed", {}).get("F1", 0) * 100
    roundBleu = evalResults.get("Open", {}).get("BLEU-1", 0)
    roundRouge = evalResults.get("Open", {}).get("ROUGE-L", 0)
    roundBert = evalResults.get("Open", {}).get("BERTScore", 0)
    
    fullRoundMb = calculateCommunicationMetrics(totalParamsCount, numClients=3)
    cumulativeFullGb += (fullRoundMb / 1024)
    
    avgTrainingTime = sum(latestTrainTimes) / len(latestTrainTimes) if len(latestTrainTimes) > 0 else 0
    averageRoundLoss = (lossA + lossB + lossC) / 3
    
    roundLog = {
        "Round": roundIdx,
        "Avg Loss": round(averageRoundLoss, 4),
        "Overall Acc (%)": round(roundAccuracy, 2),
        "Closed Acc (%)": round(roundAccClosed, 2),
        "Closed F1 (%)": round(roundF1Closed, 2),
        "Open BLEU-1": round(roundBleu, 4),
        "Open ROUGE-L": round(roundRouge, 4),
        "Open BERTScore": round(roundBert, 4),
        "MB per Round": round(fullRoundMb, 2),
        "Avg Train Time (s)": round(avgTrainingTime, 2),
        "Peak VRAM": peakVramRecorded,
        "Cumul Data (GB)": round(cumulativeFullGb, 4)
    }
    performanceLog.append(roundLog)
    
    print(f"\n [KẾT QUẢ VÒNG {roundIdx}]")
    print(f"  + Loss Trung Bình  : {averageRoundLoss:.4f}")
    print(f"  + Overall Accuracy : {roundAccuracy:.2f}%")
    print(f"  + Closed Questions : Acc = {roundAccClosed:.2f}%, F1 = {roundF1Closed:.2f}%")
    print(f"  + Open Questions   : BLEU-1 = {roundBleu:.4f}, ROUGE-L = {roundRouge:.4f}, BERTScore = {roundBert:.4f}")
    print(f"  + Hiệu Suất        : Train = {avgTrainingTime:.2f}s | VRAM = {peakVramRecorded} | Truyền tải = {fullRoundMb:.2f} MB/Round (Tổng: {cumulativeFullGb:.4f} GB)")

    pd.DataFrame(performanceLog).to_csv(logPath, index=False)
    
    if roundIdx == maxRounds:
        torch.save(globalWeights, checkpointPath)
        print(f"\n   -> [Cập nhật] Mô hình tốt nhất đã được lưu tại Round {roundIdx}.")

# ==========================================
# 8. XUẤT BÁO CÁO TỔNG KẾT
# ==========================================
print("\n" + "="*80)
print(f" FINAL SUMMARY REPORT: FED-MVQA (SLAKE, FFT 448x448, {roundIdx} ROUNDS ĐÃ HOÀN TẤT) ")
print("="*80)

print("\n[1] PERFORMANCE & HARDWARE METRICS")
print(f"  + Trainable Parameters   : {trainableParams:,} (~{trainPct:.3f}%)")
print(f"  + Peak Reserved Memory   : {peakVramRecorded}")
print(f"  + Average Inference Time : {evalResults['avgInferenceTime']:.4f} giây/ảnh")

print("\n[2] CLINICAL EVALUATION METRICS (GLOBAL MODEL VÒNG CUỐI)")
if "Closed" in evalResults:
    print(f"  - CLOSED QUESTIONS ({evalResults['Closed']['count']} samples):")
    print(f"    * Accuracy : {evalResults['Closed']['ACC']*100:.2f}%")
    print(f"    * F1-Score : {evalResults['Closed']['F1']*100:.2f}%")
if "Open" in evalResults:
    print(f"  - OPEN QUESTIONS ({evalResults['Open']['count']} samples):")
    print(f"    * BLEU-1   : {evalResults['Open']['BLEU-1']:.4f}")
    print(f"    * ROUGE-L  : {evalResults['Open']['ROUGE-L']:.4f}")
    print(f"    * BERTScore: {evalResults['Open']['BERTScore']:.4f}")

print("\n[3] CONVERGENCE LOG CHI TIẾT")
commTable = pd.DataFrame(performanceLog)
print(commTable[['Round', 'Avg Loss', 'Overall Acc (%)', 'Closed Acc (%)', 'Open BLEU-1', 'MB per Round', 'Cumul Data (GB)', 'Avg Train Time (s)']].to_string(index=False))

print("\n" + "="*80)
print(" QUÁ TRÌNH THỰC NGHIỆM ĐÃ HOÀN TẤT THÀNH CÔNG ")
print("="*80)

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


--- KHỞI TẠO HẠ TẦNG FED-MVQA (FFT 448x448, FLOAT16 - 3 CLIENTS) ---


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

... (39 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



 [KẾT QUẢ VÒNG 1]
  + Loss Trung Bình  : 0.6241
  + Overall Accuracy : 73.60%
  + Closed Questions : Acc = 82.49%, F1 = 81.33%
  + Open Questions   : BLEU-1 = 0.6379, ROUGE-L = 0.7474, BERTScore = 0.9712
  + Hiệu Suất        : Train = 3661.35s | VRAM = 10.87 GB | Truyền tải = 25279.83 MB/Round (Tổng: 24.6873 GB)

   -> [Cập nhật] Mô hình tốt nhất đã được lưu tại Round 1.

 FINAL SUMMARY REPORT: FED-MVQA (SLAKE, FFT 448x448, 1 ROUNDS ĐÃ HOÀN TẤT) 

[1] PERFORMANCE & HARDWARE METRICS
  + Trainable Parameters   : 2,208,985,600 (~100.000%)
  + Peak Reserved Memory   : 10.87 GB
  + Average Inference Time : 0.3005 giây/ảnh

[2] CLINICAL EVALUATION METRICS (GLOBAL MODEL VÒNG CUỐI)
  - CLOSED QUESTIONS (177 samples):
    * Accuracy : 82.49%
    * F1-Score : 81.33%
  - OPEN QUESTIONS (323 samples):
    * BLEU-1   : 0.6379
    * ROUGE-L  : 0.7474
    * BERTScore: 0.9712

[3] CONVERGENCE LOG CHI TIẾT
 Round  Avg Loss  Overall Acc (%)  Closed Acc (%)  Open BLEU-1  MB per Round  Cumul Data (GB)  A